# CI/CD Carbon Emissions — Statistical Analysis

**Research:** *Greening the Pipeline: An Empirical Comparison of CI/CD Refinement Strategies and Their Carbon Impact Across Open-Source Projects*

**Projects:** HTTPie (Python, anchor), got (JavaScript), Retrofit (Java), resty (Go), Gson (Java, size-axis)

**Configurations:**
- **C1** — Baseline (no caching, no consolidation)
- **C2** — Dependency caching
- **C3** — Workflow consolidation (structurally identical to C1 for every project except HTTPie)
- **C4** — Combined (caching + consolidation + path filters)

Grid intensities used for the multi-region SCI table (Section 3.6):
Ireland 345, Germany 350, Norway 25, USA 386, Singapore 408 (all gCO₂eq/kWh).

Run this notebook after `scripts/collect_results.py` has produced `results/raw_data.csv`. All tables below state the actual `n` used — this notebook does not assume the full n=30 protocol has completed; re-run it after every `collect_results.py` refresh to update the numbers.


In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
from pathlib import Path

# Style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Grid intensity, gCO2eq/kWh (Section 3.6)
GRID_INTENSITY = {
    'Ireland':   345,
    'Germany':   350,
    'Norway':    25,
    'USA':       386,
    'Singapore': 408,
}

CONFIG_ORDER = ['C1', 'C2', 'C3', 'C4']
CONFIG_COLOURS = {'C1': '#4e79a7', 'C2': '#f28e2b', 'C3': '#59a14f', 'C4': '#e15759'}
PROJECT_ORDER = ['httpie', 'got', 'retrofit', 'resty', 'gson']
PROJECT_LABELS = {'httpie': 'HTTPie (Python)', 'got': 'got (JavaScript)',
                   'retrofit': 'Retrofit (Java)', 'resty': 'resty (Go)', 'gson': 'Gson (Java, size-axis)'}

BONFERRONI_ALPHA = 0.05 / 3   # three simultaneous comparisons (C2, C3, C4 vs C1)

print('Libraries loaded ✓')


## 1. Load Data

In [ ]:
df = pd.read_csv('../results/raw_data.csv')

df['energy_joules']    = pd.to_numeric(df['energy_joules'],    errors='coerce')
df['duration_seconds'] = pd.to_numeric(df['duration_seconds'], errors='coerce')
df = df.dropna(subset=['energy_joules', 'duration_seconds', 'config', 'project'])

print(f'Loaded {len(df)} measurement rows')
for project in PROJECT_ORDER:
    sub = df[df['project'] == project]
    if len(sub) == 0:
        print(f'  {project}: no data yet')
        continue
    run_counts = sub.groupby('config')['run_id'].nunique().reindex(CONFIG_ORDER).fillna(0).astype(int)
    print(f'  {project}: runs per config = {dict(run_counts)}')
df.head()


## 2. Descriptive Statistics

In [ ]:
desc_by_project = {}
for project in PROJECT_ORDER:
    sub = df[df['project'] == project]
    if len(sub) == 0:
        continue
    total_per_config = (
        sub.groupby('config')['energy_joules']
        .agg(['mean', 'median', 'std', 'count'])
        .rename(columns={'mean': 'mean_J', 'median': 'median_J', 'std': 'std_J', 'count': 'n_rows'})
        .round(4)
        .reindex(CONFIG_ORDER)
    )
    desc_by_project[project] = total_per_config
    print(f'=== {PROJECT_LABELS[project]}: total energy per config (all stages) ===')
    print(total_per_config.to_string())
    print()


## 3. Normality Tests (Shapiro-Wilk)

In [ ]:
print(f'{"Project":<10} {"Config":<6} {"Stage":<26} {"n":>5} {"W":>8} {"p":>10} {"Normal?":>8}')
print('-' * 80)

normality_results = {}
for project in PROJECT_ORDER:
    sub = df[df['project'] == project]
    for (config, stage), grp in sub.groupby(['config', 'stage']):
        vals = grp['energy_joules'].dropna().values
        if len(vals) < 3:
            continue
        stat, p = stats.shapiro(vals)
        normality_results[(project, config, stage)] = {'W': stat, 'p': p, 'normal': p > 0.05}
        normal_str = 'YES' if p > 0.05 else 'NO'
        print(f'{project:<10} {config:<6} {stage:<26} {len(vals):>5} {stat:>8.4f} {p:>10.4f} {normal_str:>8}')


## 4. Wilcoxon Signed-Rank Tests (C2, C3, C4 vs C1)

In [ ]:
def cliffs_delta(a, b):
    a, b = np.array(a), np.array(b)
    greater = sum(1 for ai in a for bi in b if ai > bi)
    lesser  = sum(1 for ai in a for bi in b if ai < bi)
    return (greater - lesser) / (len(a) * len(b))

def interpret_delta(d):
    ad = abs(d)
    if ad < 0.147: return 'negligible'
    if ad < 0.330: return 'small'
    if ad < 0.474: return 'medium'
    return 'large'

wilcoxon_results = {}
print(f'Bonferroni-corrected alpha = {BONFERRONI_ALPHA:.4f}\n')

for project in PROJECT_ORDER:
    sub = df[df['project'] == project]
    if len(sub) == 0:
        continue
    c1_total = sub[sub['config'] == 'C1'].groupby('run_id')['energy_joules'].sum().values
    print(f'--- {PROJECT_LABELS[project]} (n_C1={len(c1_total)}) ---')
    print(f'{"Comparison":<12} {"n_C1":>5} {"n_Cx":>5} {"stat":>10} {"p":>10} {"Sig?":>6} {"delta":>8} {"Effect":>10}')

    for config in ['C2', 'C3', 'C4']:
        cx_total = sub[sub['config'] == config].groupby('run_id')['energy_joules'].sum().values
        if len(cx_total) == 0 or len(c1_total) == 0:
            print(f'  {config} vs C1: insufficient data')
            continue
        try:
            min_len = min(len(c1_total), len(cx_total))
            if min_len < 1:
                raise ValueError
            stat, p = stats.wilcoxon(c1_total[:min_len], cx_total[:min_len])
        except ValueError:
            stat, p = stats.mannwhitneyu(c1_total, cx_total, alternative='two-sided')
        d = cliffs_delta(c1_total, cx_total)
        sig = 'YES' if p < BONFERRONI_ALPHA else 'NO'
        label = interpret_delta(d)
        pct_change = (cx_total.mean() - c1_total.mean()) / c1_total.mean() * 100
        wilcoxon_results[(project, config)] = {
            'stat': stat, 'p': p, 'significant': sig, 'cliffs_d': d,
            'effect': label, 'pct_change': pct_change,
            'n_c1': len(c1_total), 'n_cx': len(cx_total),
        }
        print(f'{config + " vs C1":<12} {len(c1_total):>5} {len(cx_total):>5} {stat:>10.4f} {p:>10.4f} {sig:>6} {d:>8.4f} {label:>10}')
    print()


## 5. SCI Score Calculation

In [ ]:
sci_rows = []
for project in PROJECT_ORDER:
    sub = df[df['project'] == project]
    if len(sub) == 0:
        continue
    for config in CONFIG_ORDER:
        run_totals = sub[sub['config'] == config].groupby('run_id')['energy_joules'].sum()
        if len(run_totals) == 0:
            continue
        mean_J = run_totals.mean()
        mean_kWh = mean_J / 3_600_000
        row = {'project': project, 'config': config, 'n': len(run_totals), 'mean_energy_J': round(mean_J, 4)}
        for region, intensity in GRID_INTENSITY.items():
            row[f'SCI_{region}_gCO2eq'] = round(mean_kWh * intensity, 6)
        sci_rows.append(row)

sci_df = pd.DataFrame(sci_rows)
print('=== SCI scores (gCO2eq per CI run), five grid regions ===')
print(sci_df.to_string(index=False))


## 6. Figure 5.1 — Mean Energy per Configuration, by Project


In [ ]:
fig, axes = plt.subplots(1, len(PROJECT_ORDER), figsize=(4 * len(PROJECT_ORDER), 4), sharey=False)
for ax, project in zip(axes, PROJECT_ORDER):
    sub = df[df['project'] == project]
    run_totals = sub.groupby(['config', 'run_id'])['energy_joules'].sum().reset_index()
    agg = run_totals.groupby('config')['energy_joules'].agg(['mean', 'std']).reindex(CONFIG_ORDER)
    bars = ax.bar(agg.index, agg['mean'], yerr=agg['std'].fillna(0), capsize=4,
                  color=[CONFIG_COLOURS.get(c, '#999') for c in agg.index],
                  edgecolor='white', linewidth=0.8, width=0.6)
    ax.set_title(PROJECT_LABELS[project], fontsize=10, fontweight='bold')
    ax.set_ylabel('Mean total energy per run (J)', fontsize=9)
    ax.tick_params(axis='both', which='major', labelsize=8)

fig.suptitle('Figure 5.1: Mean total energy per run, by configuration and project (error bars = ±1 SD)',
             fontsize=11, fontweight='bold', y=1.02)
fig.tight_layout()
out = FIGURES_DIR / 'fig5_1_mean_energy_by_project.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()


## 7. Figure 5.2 — Energy Distribution per Configuration, by Project


In [ ]:
fig, axes = plt.subplots(1, len(PROJECT_ORDER), figsize=(4 * len(PROJECT_ORDER), 4), sharey=False)
for ax, project in zip(axes, PROJECT_ORDER):
    sub = df[df['project'] == project]
    run_totals = sub.groupby(['config', 'run_id'])['energy_joules'].sum().reset_index()
    data_by_config = [run_totals[run_totals['config'] == c]['energy_joules'].dropna().values for c in CONFIG_ORDER]
    bp = ax.boxplot(data_by_config, labels=CONFIG_ORDER, patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 1.5},
                    flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5})
    for patch, config in zip(bp['boxes'], CONFIG_ORDER):
        patch.set_facecolor(CONFIG_COLOURS.get(config, '#aaa'))
        patch.set_alpha(0.75)
    ax.set_title(PROJECT_LABELS[project], fontsize=10, fontweight='bold')
    ax.set_ylabel('Total energy per run (J)', fontsize=9)
    ax.tick_params(axis='both', which='major', labelsize=8)

fig.suptitle('Figure 5.2: Per-run energy distribution underlying the Wilcoxon comparisons',
             fontsize=11, fontweight='bold', y=1.02)
fig.tight_layout()
out = FIGURES_DIR / 'fig5_2_energy_boxplot_by_project.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()


## 8. Figure 5.6 — SCI Across Five Grid Regions, C2 vs C4, by Project


In [ ]:
fig, axes = plt.subplots(1, len(PROJECT_ORDER), figsize=(4.2 * len(PROJECT_ORDER), 4.5), sharey=False)
regions = list(GRID_INTENSITY.keys())
x = np.arange(len(regions))
width = 0.38

for ax, project in zip(axes, PROJECT_ORDER):
    proj_sci = sci_df[sci_df['project'] == project].set_index('config')
    if 'C2' not in proj_sci.index or 'C4' not in proj_sci.index:
        ax.set_title(f'{PROJECT_LABELS[project]}\n(insufficient data)', fontsize=9)
        continue
    c2_vals = [proj_sci.loc['C2', f'SCI_{r}_gCO2eq'] for r in regions]
    c4_vals = [proj_sci.loc['C4', f'SCI_{r}_gCO2eq'] for r in regions]
    ax.bar(x - width/2, c2_vals, width, label='C2', color=CONFIG_COLOURS['C2'])
    ax.bar(x + width/2, c4_vals, width, label='C4', color=CONFIG_COLOURS['C4'])
    ax.set_xticks(x)
    ax.set_xticklabels(regions, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('SCI (gCO2eq/run)', fontsize=9)
    ax.set_title(PROJECT_LABELS[project], fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Figure 5.6: SCI per run across five grid regions, C2 vs C4, by project',
             fontsize=11, fontweight='bold', y=1.03)
fig.tight_layout()
out = FIGURES_DIR / 'fig5_6_sci_five_region.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()


## 9. Cross-Project Comparison (RQ2, Section 5.5)

In [ ]:
cross_project_rows = []
for project in PROJECT_ORDER:
    for config in ['C2', 'C3', 'C4']:
        r = wilcoxon_results.get((project, config))
        if r is None:
            continue
        cross_project_rows.append({
            'project': project, 'comparison': f'{config} vs C1',
            'pct_change': round(r['pct_change'], 2),
            'cliffs_d': round(r['cliffs_d'], 4),
            'effect': r['effect'], 'significant': r['significant'],
            'n_c1': r['n_c1'], 'n_cx': r['n_cx'],
        })

cross_df = pd.DataFrame(cross_project_rows)
print('=== Cross-project comparison: % energy change and effect size, by project ===')
print(cross_df.to_string(index=False))
print()
print('C1-vs-C3 rows close to 0% with negligible/small effect confirm the internal-consistency')
print('check for got, Retrofit, resty, and Gson (Section 3.2.5): these projects have nothing to')
print('consolidate, so C1 and C3 should show no systematic difference beyond measurement noise.')


## 10. Paper-Ready Results Summary

In [ ]:
print('=' * 70)
print('  RESULTS SUMMARY -- CI/CD Carbon Emissions Study')
print('  Ready to copy into dissertation Chapter 5')
print('=' * 70)

for project in PROJECT_ORDER:
    sub = df[df['project'] == project]
    if len(sub) == 0:
        continue
    print(f'\n--- {PROJECT_LABELS[project]} ---')
    print(f'{"Config":<8} {"Mean (J)":>10} {"Median (J)":>12} {"SD (J)":>10} {"n_runs":>8}')
    print('-' * 52)
    for config in CONFIG_ORDER:
        run_totals = sub[sub['config'] == config].groupby('run_id')['energy_joules'].sum()
        if len(run_totals) == 0:
            continue
        print(f'{config:<8} {run_totals.mean():>10.4f} {run_totals.median():>12.4f} {run_totals.std():>10.4f} {len(run_totals):>8}')

print('\n--- Wilcoxon tests vs C1 (Bonferroni alpha = 0.05/3), all projects ---')
print(cross_df.to_string(index=False))

print('\n--- SCI scores, five regions, all projects and configs ---')
print(sci_df.to_string(index=False))

print('\n' + '=' * 70)
print('NOTE: figures above state the actual n achieved as of this run.')
print('Re-run this notebook after each collect_results.py refresh to update all numbers.')
print('=' * 70)
